# RFP Mistral Workflow

This notebook mirrors the Python modules in `src/rfp_mistral/` so you can run the full RFP pipeline interactively.

## Step 1: Environment Setup

Run this only once per environment.

In [1]:
%pip install -e ..


Obtaining file:///Users/abhinavkaushik/Documents/RFP_LLM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rfp-mistral (pyproject.toml) ... done
  Created wheel for rfp-mistral: filename=rfp_mistral-0.1.0-0.editable-py3-none-any.whl size=3199 sha256=d0e8ebc8d143751c475b7a9ea868e4446d8dc688cd07bfd8388b9d4d16758217
  Stored in directory: /private/var/folders/l1/lrszwpw11nb22fb9wx2vph_w0000gn/T/pip-ephem-wheel-cache-nj2y1x2y/wheels/5c/76/c6/8032f5a51cafd929036db862b322bafc1f2bda4d06e987a7d8
Successfully built rfp-mistral
  Attempting uninstall: rfp-mistral
    Found existing installation: rfp-mistral 0.1.0
    Uninstalling rfp-mistral-0.1.0:
      Successfully uninstalled rfp-mistral-0.1.0

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you m

## Step 2: Project Setup

In [1]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from textwrap import dedent
from typing import Any

import faiss
import numpy as np
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = ROOT / "data" / "raw" / "rfp_records.jsonl"
SAMPLE_DATA_PATH = ROOT / "data" / "samples" / "rfp_records.sample.jsonl"
TRAIN_DATA_PATH = ROOT / "data" / "processed" / "train.jsonl"
INDEX_DIR = ROOT / "data" / "index"
CHECKPOINT_DIR = ROOT / "checkpoints" / "rfp-mistral-lora"
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

ULTRA_PROPOSALS_PATH = Path.home() / "Downloads" / "final_ultra_proposals_100.json"
TONE_VISUALS_PATH = Path.home() / "Downloads" / "ultimate_proposals_with_tone_visuals(1).json"
CLIENT_INDUSTRY_MAP = {
    "AT&T US": "Telecommunications",
    "Airtel India": "Telecommunications",
    "BT Group UK": "Telecommunications",
    "Etisalat UAE": "Telecommunications",
    "MTN Africa": "Telecommunications",
    "Orange France": "Telecommunications",
    "Reliance Jio": "Telecommunications",
    "T-Mobile US": "Telecommunications",
    "Telefonica Spain": "Telecommunications",
    "Vodafone EU": "Telecommunications",
}


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3: Define The RFP Data Structure

In [2]:
@dataclass
class RFPRecord:
    id: str
    client_name: str
    industry: str
    rfp_title: str
    rfp_summary: str
    requirements: list[str] = field(default_factory=list)
    evaluation_criteria: list[str] = field(default_factory=list)
    solution_summary: str = ""
    proposal_sections: dict[str, str] = field(default_factory=dict)
    tags: list[str] = field(default_factory=list)

    @classmethod
    def from_dict(cls, payload: dict[str, Any]) -> "RFPRecord":
        return cls(
            id=str(payload["id"]),
            client_name=str(payload.get("client_name", "")),
            industry=str(payload.get("industry", "")),
            rfp_title=str(payload.get("rfp_title", "")),
            rfp_summary=str(payload.get("rfp_summary", "")),
            requirements=[str(item) for item in payload.get("requirements", [])],
            evaluation_criteria=[str(item) for item in payload.get("evaluation_criteria", [])],
            solution_summary=str(payload.get("solution_summary", "")),
            proposal_sections={str(key): str(value) for key, value in payload.get("proposal_sections", {}).items()},
            tags=[str(item) for item in payload.get("tags", [])],
        )


@dataclass
class TrainingExample:
    record_id: str
    prompt: str
    response: str
    target_section: str

    def to_dict(self) -> dict[str, str]:
        return {
            "record_id": self.record_id,
            "prompt": self.prompt,
            "response": self.response,
            "target_section": self.target_section,
            "text": f"{self.prompt}{self.response}",
        }


def load_rfp_records(path: str | Path) -> list[RFPRecord]:
    records: list[RFPRecord] = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            records.append(RFPRecord.from_dict(json.loads(line)))
    return records


def write_jsonl(path: str | Path, rows: list[dict]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=True) + "\n")


def read_json_array(path: str | Path) -> list[dict[str, Any]]:
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(payload, list):
        raise ValueError(f"Expected a JSON array in {path}")
    return [item for item in payload if isinstance(item, dict)]


def labelize(text: str) -> str:
    return text.replace("_", " ").strip().title()


def slugify(text: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")
    return slug or "record"


def stringify_scalar(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, bool):
        return "Yes" if value else "No"
    return str(value)


def flatten_kv_items(payload: dict[str, Any]) -> list[str]:
    lines: list[str] = []
    seen_lines: set[str] = set()

    def append_line(line: str) -> None:
        if line and line not in seen_lines:
            seen_lines.add(line)
            lines.append(line)

    for key, value in payload.items():
        label = labelize(key)
        if isinstance(value, dict):
            nested = ", ".join(flatten_kv_items(value))
            if nested:
                append_line(f"{label}: {nested}")
        elif isinstance(value, list):
            rendered_items: list[str] = []
            seen_items: set[str] = set()
            for item in value:
                if item is None:
                    continue
                rendered_item = stringify_scalar(item).strip()
                if rendered_item and rendered_item not in seen_items:
                    seen_items.add(rendered_item)
                    rendered_items.append(rendered_item)
            rendered = ", ".join(rendered_items)
            if rendered:
                append_line(f"{label}: {rendered}")
        else:
            rendered = stringify_scalar(value).strip()
            if rendered:
                append_line(f"{label}: {rendered}")
    return lines


def format_section_value(value: Any) -> str:
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, dict):
        lines = flatten_kv_items(value)
        return "\n".join(f"- {line}" for line in lines)
    if isinstance(value, list):
        return "\n".join(f"- {stringify_scalar(item)}" for item in value if item is not None)
    return stringify_scalar(value).strip()


def build_requirements(record: dict[str, Any]) -> list[str]:
    requirements: list[str] = []
    for field in ("client_kpis", "delivery", "contract_annexure", "visual_assets", "negotiation_hooks"):
        value = record.get(field)
        if isinstance(value, dict):
            requirements.extend(flatten_kv_items(value))
    if isinstance(record.get("financials"), dict):
        requirements.extend(flatten_kv_items(record["financials"]))
    if isinstance(record.get("roi_summary"), dict):
        requirements.extend(flatten_kv_items(record["roi_summary"]))
    return requirements


def build_evaluation_criteria(record: dict[str, Any]) -> list[str]:
    criteria = [
        "Business impact and ROI clarity",
        "Technical fit for telecom operations",
        "Delivery feasibility and timeline confidence",
        "Commercial flexibility and implementation value",
    ]
    if record.get("competitor_positioning"):
        criteria.append("Differentiation against incumbent vendors")
    if record.get("visual_assets"):
        criteria.append("Executive-friendly presentation and visual storytelling")
    return criteria


def build_sections(record: dict[str, Any], source: str) -> dict[str, str]:
    field_map = {
        "executive_summary": "executive_summary",
        "tone_specific_summary": "tone_specific_summary",
        "deal_storytelling": "deal_storytelling",
        "competitor_positioning": "competitor_positioning",
        "architecture_diagram_text": "architecture_diagram",
        "financials": "financials",
        "pricing": "pricing_notes",
        "contract_annexure": "contract_annexure",
        "delivery": "implementation_plan",
        "visual_assets": "visual_assets",
        "negotiation_hooks": "negotiation_hooks",
        "roi_summary": "roi_summary",
    }
    sections: dict[str, str] = {}
    for source_key, target_key in field_map.items():
        value = record.get(source_key)
        if value in (None, "", [], {}):
            continue
        rendered = format_section_value(value)
        if rendered:
            sections[target_key] = rendered

    if "executive_summary" not in sections and "tone_specific_summary" in sections:
        sections["executive_summary"] = sections["tone_specific_summary"]
    if "pricing_notes" not in sections and "financials" in sections:
        sections["pricing_notes"] = sections["financials"]

    sections["source_dataset"] = source
    return sections


def build_summary(record: dict[str, Any], sections: dict[str, str]) -> str:
    parts = [
        stringify_scalar(record.get("executive_summary")),
        stringify_scalar(record.get("tone_specific_summary")),
        stringify_scalar(record.get("deal_storytelling")),
    ]
    summary = " ".join(part.strip() for part in parts if part and part.strip())
    if summary:
        return summary
    for key in ("executive_summary", "tone_specific_summary", "deal_storytelling"):
        if key in sections:
            return sections[key]
    return "Telecom transformation proposal focused on measurable operational outcomes."


def normalize_proposal_record(record: dict[str, Any], source: str, sequence: int) -> RFPRecord:
    client_name = stringify_scalar(record.get("client")) or "Unknown Client"
    sections = build_sections(record, source)
    summary = build_summary(record, sections)
    requirements = build_requirements(record)

    return RFPRecord(
        id=f"{slugify(source)}-{sequence:03d}",
        client_name=client_name,
        industry=CLIENT_INDUSTRY_MAP.get(client_name, "Telecommunications"),
        rfp_title=f"AIOps Transformation Proposal for {client_name}",
        rfp_summary=summary,
        requirements=requirements,
        evaluation_criteria=build_evaluation_criteria(record),
        solution_summary=sections.get("executive_summary", summary),
        proposal_sections=sections,
        tags=[
            "telecommunications",
            "aiops",
            "proposal",
            slugify(client_name),
            slugify(source),
        ],
    )


def build_training_examples(records: list[RFPRecord], target_section: str = "all") -> list[TrainingExample]:
    examples: list[TrainingExample] = []
    for record in records:
        if target_section == "all":
            sections = sorted(record.proposal_sections)
        else:
            sections = [target_section] if target_section in record.proposal_sections else []
        examples.extend(build_training_example(record, section) for section in sections)
    return examples


## Step 4: Build The Training Prompt

In [3]:
SYSTEM_PROMPT = (
    "You are an enterprise proposal specialist. You understand user intent and produce accurate, persuasive, "
    "and requirement-aligned proposal responses."
)


INTENT_GUIDANCE = {
    "generate_proposal": "Create a full proposal draft using the problem statement, requirements, and retrieved examples.",
    "generate_section": "Create only the requested proposal section.",
    "revise_proposal": "Revise the existing proposal while preserving the core solution and improving the requested areas.",
    "revise_section": "Revise only the requested section while preserving factual accuracy.",
    "proposal_to_json": "Return the answer as structured JSON instead of prose.",
}


DEFAULT_REVISION_INSTRUCTION = (
    "Improve clarity, executive tone, structure, and alignment to the business problem without inventing new facts."
)


FULL_SECTION_ORDER = [
    "executive_summary",
    "deal_storytelling",
    "competitor_positioning",
    "architecture_diagram",
    "implementation_plan",
    "financials",
    "pricing_notes",
    "contract_annexure",
    "visual_assets",
    "negotiation_hooks",
    "roi_summary",
]


SECTION_LABELS = {
    "executive_summary": "Executive Summary",
    "deal_storytelling": "Deal Storytelling",
    "competitor_positioning": "Competitor Positioning",
    "architecture_diagram": "Architecture Diagram Narrative",
    "implementation_plan": "Implementation Plan",
    "financials": "Financials",
    "pricing_notes": "Pricing Notes",
    "contract_annexure": "Contract Annexure",
    "visual_assets": "Visual Assets",
    "negotiation_hooks": "Negotiation Hooks",
    "roi_summary": "ROI Summary",
}


OUTPUT_FORMAT_GUIDANCE = {
    "plain_english": "Return plain business English. Do not use JSON unless explicitly asked.",
    "json": "Return valid JSON only.",
}


REVISION_REQUESTS = {
    "executive_summary": "Revise the executive summary to sound more executive-ready and outcome-focused.",
    "implementation_plan": "Revise the implementation plan to be clearer on phases, ownership, and delivery confidence.",
    "pricing_notes": "Revise the pricing notes to be clearer and easier for commercial review.",
    "contract_annexure": "Revise the contract annexure to improve readability and governance clarity.",
}


JSON_SCHEMA_HINT = {
    "proposal_json": {
        "client_name": "string",
        "industry": "string",
        "rfp_title": "string",
        "problem_statement": "string",
        "requirements": ["string"],
        "evaluation_criteria": ["string"],
        "proposal_sections": {
            "executive_summary": "string",
            "implementation_plan": "string",
            "pricing_notes": "string"
        }
    },
    "section_json": {
        "client_name": "string",
        "industry": "string",
        "rfp_title": "string",
        "target_section": "string",
        "content": "string"
    }
}


def format_bullets(items: list[str]) -> str:
    if not items:
        return "- None provided"
    return "\n".join(f"- {item}" for item in items)



def build_section_text(record: RFPRecord, target_section: str = "all") -> str:
    if target_section == "all":
        section_names = [name for name in FULL_SECTION_ORDER if name in record.proposal_sections]
        section_names.extend(
            name for name in sorted(record.proposal_sections)
            if name not in section_names and name != "source_dataset"
        )
        blocks = []
        for section_name in section_names:
            blocks.append(
                f"{SECTION_LABELS.get(section_name, labelize(section_name))}:\n{record.proposal_sections[section_name]}"
            )
        return "\n\n".join(blocks)

    return record.proposal_sections[target_section]



def build_json_response(record: RFPRecord, target_section: str = "all") -> str:
    if target_section == "all":
        payload = {
            "client_name": record.client_name,
            "industry": record.industry,
            "rfp_title": record.rfp_title,
            "problem_statement": record.rfp_summary,
            "requirements": record.requirements,
            "evaluation_criteria": record.evaluation_criteria,
            "proposal_sections": {
                key: value
                for key, value in record.proposal_sections.items()
                if key != "source_dataset"
            },
        }
    else:
        payload = {
            "client_name": record.client_name,
            "industry": record.industry,
            "rfp_title": record.rfp_title,
            "target_section": target_section,
            "content": record.proposal_sections[target_section],
        }
    return json.dumps(payload, indent=2, ensure_ascii=True)



def build_response_text(
    record: RFPRecord,
    intent: str,
    output_format: str,
    target_section: str,
) -> str:
    if output_format == "json" or intent == "proposal_to_json":
        return build_json_response(record, target_section)
    return build_section_text(record, target_section)



def build_user_request(intent: str, output_format: str, target_section: str) -> str:
    if intent == "generate_proposal":
        if output_format == "json":
            return "Generate a complete proposal response and return it in JSON."
        return "Generate a complete proposal response in plain business English."

    if intent == "generate_section":
        label = SECTION_LABELS.get(target_section, labelize(target_section))
        if output_format == "json":
            return f"Generate the {label} section and return the answer as JSON."
        return f"Generate the {label} section in plain business English."

    if intent == "revise_proposal":
        if output_format == "json":
            return "Revise the current proposal and return the updated proposal in JSON."
        return "Revise the current proposal and return the improved version in plain business English."

    if intent == "revise_section":
        label = SECTION_LABELS.get(target_section, labelize(target_section))
        if output_format == "json":
            return f"Revise the current {label} section and return the updated version as JSON."
        return f"Revise the current {label} section in plain business English."

    if intent == "proposal_to_json":
        return "Convert this proposal into JSON."

    raise ValueError(f"Unsupported intent: {intent}")



def build_instruction_prompt(
    record: RFPRecord,
    target_section: str,
    intent: str = "generate_section",
    output_format: str = "plain_english",
    user_request: str | None = None,
    revision_instruction: str | None = None,
) -> str:
    request_text = user_request or build_user_request(intent, output_format, target_section)
    current_draft = ""
    if intent in {"revise_proposal", "revise_section"}:
        current_draft = (
            "\nCurrent Draft To Revise:\n"
            f"{build_section_text(record, target_section)}\n"
            "\nRevision Goal:\n"
            f"{revision_instruction or DEFAULT_REVISION_INSTRUCTION}\n"
        )

    json_schema_text = ""
    if output_format == "json" or intent == "proposal_to_json":
        schema_hint = JSON_SCHEMA_HINT["proposal_json" if target_section == "all" else "section_json"]
        json_schema_text = (
            "\nJSON Output Schema Hint:\n"
            f"{json.dumps(schema_hint, indent=2, ensure_ascii=True)}\n"
        )

    return dedent(
        f"""<s>[INST] {SYSTEM_PROMPT}

Intent: {intent}
Intent Guidance: {INTENT_GUIDANCE[intent]}
Output Format: {output_format}
Output Guidance: {OUTPUT_FORMAT_GUIDANCE[output_format]}
Target Section: {target_section}

User Request:
{request_text}

Client: {record.client_name}
Industry: {record.industry}
RFP Title: {record.rfp_title}

Problem Statement:
{record.rfp_summary}

Requirements:
{format_bullets(record.requirements)}

Evaluation Criteria:
{format_bullets(record.evaluation_criteria)}

Known Solution Direction:
{record.solution_summary or 'Not provided'}{current_draft}{json_schema_text}
Instructions:
- Follow the requested intent exactly
- Use only the information supported by the record
- Keep the answer aligned to the listed requirements
- If output format is plain_english, return normal business prose
- If output format is json, return valid JSON only
[/INST]
"""
    )



def build_training_example(
    record: RFPRecord,
    target_section: str,
    intent: str = "generate_section",
    output_format: str = "plain_english",
    user_request: str | None = None,
    revision_instruction: str | None = None,
) -> TrainingExample:
    return TrainingExample(
        record_id=record.id,
        prompt=build_instruction_prompt(
            record,
            target_section=target_section,
            intent=intent,
            output_format=output_format,
            user_request=user_request,
            revision_instruction=revision_instruction,
        ),
        response=build_response_text(record, intent, output_format, target_section) + "</s>",
        target_section=target_section,
    )



## Step 5: Load And Merge Proposal Data

This notebook reads both sample proposal JSON files from `Downloads`, normalizes them into the `RFPRecord` schema, saves the merged raw JSONL to `RAW_DATA_PATH`, and keeps the merged records in memory for the remaining workflow.


In [5]:
ultra_records = read_json_array(ULTRA_PROPOSALS_PATH)
tone_visual_records = read_json_array(TONE_VISUALS_PATH)

records = [
    *[
        normalize_proposal_record(record, "final_ultra_proposals_100", index)
        for index, record in enumerate(ultra_records, start=1)
    ],
    *[
        normalize_proposal_record(record, "ultimate_proposals_with_tone_visuals", index)
        for index, record in enumerate(tone_visual_records, start=1)
    ],
]

write_jsonl(RAW_DATA_PATH, [record.__dict__ for record in records])
print(f"Saved {len(records)} merged raw records to {RAW_DATA_PATH}")
len(records), records[0].rfp_title


Saved 200 merged raw records to /Users/abhinavkaushik/Documents/RFP_LLM/data/raw/rfp_records.jsonl


(200, 'AIOps Transformation Proposal for Reliance Jio')

## Step 6: Create The Fine-Tuning Dataset

This dataset now teaches the model multiple intents from the same proposal records: generate a full proposal, generate a section, revise an existing draft, and return either plain English or JSON depending on the request.


In [6]:
intent_configs = [
    {"intent": "generate_proposal", "output_format": "plain_english", "target_section": "all"},
    {"intent": "generate_proposal", "output_format": "json", "target_section": "all"},
    {"intent": "proposal_to_json", "output_format": "json", "target_section": "all"},
]

for section_name in [name for name in FULL_SECTION_ORDER if any(name in record.proposal_sections for record in records)]:
    intent_configs.append(
        {"intent": "generate_section", "output_format": "plain_english", "target_section": section_name}
    )
    intent_configs.append(
        {"intent": "generate_section", "output_format": "json", "target_section": section_name}
    )
    if section_name in REVISION_REQUESTS:
        intent_configs.append(
            {
                "intent": "revise_section",
                "output_format": "plain_english",
                "target_section": section_name,
                "revision_instruction": REVISION_REQUESTS[section_name],
            }
        )

intent_configs.append(
    {
        "intent": "revise_proposal",
        "output_format": "plain_english",
        "target_section": "all",
        "revision_instruction": "Revise the full proposal so it reads like a stronger executive response while preserving the same solution and facts.",
    }
)

examples: list[TrainingExample] = []
for record in records:
    for config in intent_configs:
        target_section = config["target_section"]
        if target_section != "all" and target_section not in record.proposal_sections:
            continue
        examples.append(
            build_training_example(
                record,
                target_section=target_section,
                intent=config["intent"],
                output_format=config["output_format"],
                revision_instruction=config.get("revision_instruction"),
            )
        )

write_jsonl(TRAIN_DATA_PATH, [example.to_dict() for example in examples])
print(f"Saved {len(examples)} intent-aware training examples to {TRAIN_DATA_PATH}")
examples[0].to_dict()



Saved 1600 training examples to /Users/abhinavkaushik/Documents/RFP_LLM/data/processed/train.jsonl


{'record_id': 'final-ultra-proposals-100-001',
 'prompt': '<s>[INST] You are an enterprise proposal specialist. Produce accurate, persuasive, and requirement-aligned RFP response sections.\n\nGenerate the `architecture_diagram` section for the following RFP.\n\nClient: Reliance Jio\nIndustry: Telecommunications\nRFP Title: AIOps Transformation Proposal for Reliance Jio\n\nRFP Summary:\nTailored AIOps transformation for Reliance Jio addressing operational inefficiencies. Reliance Jio operates a complex multi-vendor network environment involving Nokia, Huawei, Ericsson leading to fragmented visibility and delayed response.\n\nRequirements:\n- Current Mttr: 100 minutes\n- Target Mttr: 20 minutes\n- Current Alerts Per Day: 2086996\n- Target Alert Reduction: 63%\n- Network Availability: 99.75%\n- Timeline: 5 months\n- Team: Architect, ML Engineer, DevOps, Telecom SME\n- Sla: 99.99% uptime guarantee, Critical incident response within 15 minutes\n- Penalties: 5% monthly fee penalty for SLA br

## Step 7: Build The Proposal Retrieval Memory

In [7]:
def build_document(record: RFPRecord) -> str:
    sections = "\n".join(
        f"{name}: {text}" for name, text in record.proposal_sections.items()
    )
    requirements = "\n".join(f"- {item}" for item in record.requirements)
    return (
        f"Client: {record.client_name}\n"
        f"Industry: {record.industry}\n"
        f"Title: {record.rfp_title}\n"
        f"Summary: {record.rfp_summary}\n"
        f"Requirements:\n{requirements}\n"
        f"Proposal:\n{sections}"
    )


embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
documents = [build_document(record) for record in records]
embedding_model = SentenceTransformer(embedding_model_name)
embeddings = embedding_model.encode(documents, normalize_embeddings=True)
matrix = np.asarray(embeddings, dtype=np.float32)

INDEX_DIR.mkdir(parents=True, exist_ok=True)
index = faiss.IndexFlatIP(matrix.shape[1])
index.add(matrix)
faiss.write_index(index, str(INDEX_DIR / "proposal.index"))

write_jsonl(INDEX_DIR / "metadata.jsonl", [record.__dict__ for record in records])
with (INDEX_DIR / "manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "embedding_model": embedding_model_name,
            "dimension": int(matrix.shape[1]),
            "count": int(matrix.shape[0]),
        },
        handle,
        indent=2,
    )

print(f"Saved retrieval index to {INDEX_DIR}")


Saved retrieval index to /Users/abhinavkaushik/Documents/RFP_LLM/data/index


## Step 8: Find Similar Historical Proposals

In [8]:
def search_proposals(query: str, top_k: int = 3) -> list[dict]:
    with (INDEX_DIR / "manifest.json").open("r", encoding="utf-8") as handle:
        manifest = json.load(handle)
    with (INDEX_DIR / "metadata.jsonl").open("r", encoding="utf-8") as handle:
        metadata = [json.loads(line) for line in handle if line.strip()]

    model = SentenceTransformer(manifest["embedding_model"])
    index = faiss.read_index(str(INDEX_DIR / "proposal.index"))
    embedding = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.asarray(embedding, dtype=np.float32), top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        item = dict(metadata[idx])
        item["retrieval_score"] = float(score)
        results.append(item)
    return results


retrieved = search_proposals("healthcare cloud migration executive summary", top_k=2)
retrieved


[{'id': 'ultimate-proposals-with-tone-visuals-094',
  'client_name': 'T-Mobile US',
  'industry': 'Telecommunications',
  'rfp_title': 'AIOps Transformation Proposal for T-Mobile US',
  'rfp_summary': 'T-Mobile US demands high-performance, low-latency AIOps systems with enterprise-grade scalability. T-Mobile US is transitioning towards AI-driven operations due to increasing network complexity.',
  'requirements': ['Architecture Diagram: Data Sources → Kafka → Processing (Flink) → ML Models → RCA Engine → Dashboard',
   'Roi Chart: Bar chart showing cost vs savings over 12 months',
   'Transformation Roadmap: Phase 1: Assessment, Phase 2: Pilot, Phase 3: Scale, Phase 4: Optimization',
   'Optional Modules: Advanced RCA Engine, Auto-remediation Suite, AI Forecasting Module',
   'Discount Tiers: 5% discount for 3-year contract, 10% discount for enterprise-wide rollout',
   'Strategic Language: Positioned as a long-term strategic transformation partner with co-innovation opportunities.',
 

## Step 9: Build The Final Generation Request

This step now mirrors the same intent-aware behavior used for training. You can ask for a fresh proposal, request a revision, or choose plain English versus JSON output.



In [9]:
def build_generation_request(
    problem_statement: str,
    retrieved_examples: list[dict],
    intent: str = "generate_proposal",
    output_format: str = "plain_english",
    target_section: str = "all",
    current_draft: str = "",
    revision_instruction: str = "",
) -> str:
    examples_block = []
    for item in retrieved_examples:
        sections = item.get("proposal_sections", {})
        proposal_snapshot = []
        for section_name in FULL_SECTION_ORDER:
            if section_name in sections and section_name != "source_dataset":
                proposal_snapshot.append(
                    f"{SECTION_LABELS.get(section_name, labelize(section_name))}:\n{sections[section_name]}"
                )
        examples_block.append(
            dedent(
                f"""Example Proposal
Client: {item.get('client_name', '')}
Industry: {item.get('industry', '')}
Title: {item.get('rfp_title', '')}
Relevant Requirements:
{chr(10).join(f'- {req}' for req in item.get('requirements', []))}
Proposal Snapshot:
{chr(10).join(proposal_snapshot[:3])}
"""
            )
        )

    joined_examples = "\n\n".join(examples_block) if examples_block else "No retrieved examples."

    revision_block = ""
    if intent in {"revise_proposal", "revise_section"} and current_draft.strip():
        revision_block = (
            f"\nCurrent Draft To Revise:\n{current_draft}\n"
            f"\nRevision Goal:\n{revision_instruction or DEFAULT_REVISION_INSTRUCTION}\n"
        )

    json_block = ""
    if output_format == "json" or intent == "proposal_to_json":
        schema_hint = JSON_SCHEMA_HINT["proposal_json" if target_section == "all" else "section_json"]
        json_block = f"\nReturn valid JSON only. Use this schema shape as a guide:\n{json.dumps(schema_hint, indent=2, ensure_ascii=True)}\n"

    return dedent(
        f"""You are preparing a high-quality RFP response.

Intent: {intent}
Output Format: {output_format}
Target Section: {target_section}

Problem Statement:
{problem_statement}

Retrieved Historical Examples:
{joined_examples}{revision_block}{json_block}
Write the best possible response for the requested intent. If no output format is specified, default to plain business English.
"""
    )


problem_statement = "Reliance Jio operates a complex multi-vendor network environment involving Nokia, Huawei, and Ericsson, creating fragmented visibility, delayed incident response, and a need to reduce MTTR from 100 minutes to 20 minutes within 5 months."
intent = "generate_proposal"
output_format = "plain_english"
target_section = "all"
current_draft = ""
revision_instruction = ""

retrieved = search_proposals(problem_statement, top_k=2)
prompt = build_generation_request(
    problem_statement=problem_statement,
    retrieved_examples=retrieved,
    intent=intent,
    output_format=output_format,
    target_section=target_section,
    current_draft=current_draft,
    revision_instruction=revision_instruction,
)
print(prompt)



You are preparing a high-quality RFP response.

User Request:
Draft an executive summary for a healthcare cloud migration RFP with managed services

Retrieved Historical Examples:
Example Proposal
Client: T-Mobile US
Industry: Telecommunications
Title: AIOps Transformation Proposal for T-Mobile US
Relevant Requirements:
- Architecture Diagram: Data Sources → Kafka → Processing (Flink) → ML Models → RCA Engine → Dashboard
- Roi Chart: Bar chart showing cost vs savings over 12 months
- Transformation Roadmap: Phase 1: Assessment, Phase 2: Pilot, Phase 3: Scale, Phase 4: Optimization
- Optional Modules: Advanced RCA Engine, Auto-remediation Suite, AI Forecasting Module
- Discount Tiers: 5% discount for 3-year contract, 10% discount for enterprise-wide rollout
- Strategic Language: Positioned as a long-term strategic transformation partner with co-innovation opportunities.
- Savings: $5M annually
- Payback: 11 months
Executive Summary:
T-Mobile US demands high-performance, low-latency AIOp

## Step 10: Fine-Tune Mistral

Set `use_4bit=True` if you want quantized loading for QLoRA.

In [10]:
use_4bit = True
dataset = load_dataset("json", data_files=str(TRAIN_DATA_PATH), split="train")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "device_map": "auto",
    "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
}

if use_4bit:
    from transformers import BitsAndBytesConfig

    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    learning_rate=2e-4,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_ratio=0.05,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    peft_config=lora_config,
    args=training_args,
)

# Uncomment when you are ready to train.
# trainer.train()
# trainer.model.save_pretrained(CHECKPOINT_DIR)
# tokenizer.save_pretrained(CHECKPOINT_DIR)


Generating train split: 1600 examples [00:00, 46890.58 examples/s]


PackageNotFoundError: No package metadata was found for bitsandbytes

## Step 11: Generate A Proposal Draft

In [ ]:
adapter_path = CHECKPOINT_DIR if CHECKPOINT_DIR.exists() else None

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

if adapter_path:
    model = PeftModel.from_pretrained(model, str(adapter_path))

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


## Step 12: Prepare This For Production

- Replace the sample JSONL with your real RFP and proposal corpus.
- Generate one training dataset per section type, or add section labels to a larger mixed dataset.
- Add a compliance checker that verifies the generated answer covers every requirement.
- Add redaction before training if your proposals contain sensitive information.